# Time discretization schemes (Euler, Milstein, exact GBM)

**Start here:** This deep dive expands on `07_advanced_quant/monte_carlo_simulation.ipynb`; read the overview first for the canonical pricer workflow.

How **approximate schemes** relate to the **exact GBM** stepper exposed in Python, and how to study **accuracy vs cost** when you can only tune **`num_steps`** and **`num_paths`** at the high level.


## Concept

**Euler–Maruyama** and related schemes approximate SDEs on a grid; error usually scales with the step size for smooth payoffs. **Exact GBM** sampling matches the **one-step lognormal** law over each interval, so **European** prices under GBM should not exhibit discretization bias in $ \Delta t $. Python `EuropeanPricer` uses `ExactGbm` on the Rust side.

Scheme selectors (Euler–Maruyama, log-Euler, Milstein, exact GBM) live purely inside the Rust crate `finstack-quant-models` and are not exposed as Python handles today. Python users study **accuracy vs cost** only through `num_paths` and `num_steps`.

## API walkthrough

Discretisation schemes are Rust-internal; the Python surface exposes canonical product pricers.

In [ ]:
import finstack_quant.models.monte_carlo as mc

print("Public Python MC entry points for GBM pricing:")
for name in ("EuropeanPricer", "PathDependentPricer", "LsmcPricer"):
    cls = getattr(mc, name)
    print(f"  {name}: {cls!r}")

print(
    "\nScheme handles (ExactGbm, EulerMaruyama, LogEuler, Milstein) are "
    "Rust-internal and intentionally not exposed as Python classes."
)


## Examples

Use **`EuropeanPricer`** with different step counts for the same European call. With exact GBM the mean should track Black–Scholes regardless of `num_steps`, while runtime grows with more steps.

Tables: ATM inputs; **step count** mainly affects **runtime** under **exact GBM**, while **path count** controls **standard error** and **distance to the analytical** price.

In [ ]:
from finstack_quant.models import bs_price
from finstack_quant.models.monte_carlo import EuropeanPricer

spot, strike, rate, q, vol, T = 100.0, 100.0, 0.05, 0.0, 0.20, 1.0
bs = bs_price(spot, strike, rate, q, vol, T, True)
print(f"Black–Scholes call (anchor): {bs:.6f}")

pricer = EuropeanPricer(num_paths=40_000, seed=7)
step_counts = [1, 8, 32, 128, 512]
print("num_steps | MC_mean | stderr | |error vs BS|")
for n in step_counts:
    res = pricer.price_call(spot, strike, rate, q, vol, T, num_steps=n)
    m = res.mean.amount
    err = abs(m - bs)
    print(f"{n:9d} | {m:.6f} | {res.stderr:.6f} | {err:.6f}")

In [ ]:
from finstack_quant.models import bs_price
from finstack_quant.models.monte_carlo import EuropeanPricer

spot, strike, rate, q, vol, T = 100.0, 100.0, 0.05, 0.0, 0.20, 1.0
bs = bs_price(spot, strike, rate, q, vol, T, True)
path_counts = [1_000, 5_000, 10_000, 50_000, 100_000]
print("num_paths | MC_mean | stderr | |error vs BS|")
for n in path_counts:
    pr = EuropeanPricer(num_paths=n, seed=99)
    r = pr.price_call(spot, strike, rate, q, vol, T, num_steps=1)
    m = r.mean.amount
    print(f"{n:9d} | {m:.6f} | {r.stderr:.6f} | {abs(m - bs):.6f}")

### Four times the paths: what actually halves?

For independent payoffs of finite variance, $\operatorname{SE}(\bar X)=s/\sqrt{N}$. Use several independent seeds to compare root-mean-square error and average estimated SE at $N$ and $4N$. Their expected scale is halved; the realized error of an individual run is not guaranteed to halve. Changing the exact-GBM time grid also changes the random-number consumption, so it does not couple the two path sets automatically.

In [ ]:
import math
import statistics

reference = bs_price(100, 100, 0.05, 0.0, 0.2, 1.0, True)
scaling = []
for count in (1024, 4096):
    runs = [EuropeanPricer(count, seed, False).price_call(
        100, 100, 0.05, 0.0, 0.2, 1.0, num_steps=1
    ) for seed in range(16)]
    average_se = statistics.mean(run.stderr for run in runs)
    rmse = math.sqrt(statistics.mean((run.mean.amount - reference) ** 2 for run in runs))
    scaling.append((average_se, rmse))
    print(f"N={count}: average SE={average_se:.6f}, independent-seed RMSE={rmse:.6f}")
se_ratio = scaling[1][0] / scaling[0][0]
print(f"SE(4N)/SE(N)={se_ratio:.4f}; theoretical scale=0.5")
print(f"Observed RMSE ratio={scaling[1][1] / scaling[0][1]:.4f}")
# A broad sampling diagnostic on this fixed seed panel, not a pathwise identity.
assert 0.4 < se_ratio < 0.65

## Takeaways

- Prefer **exact transitions** for GBM when available: **no discretization bias** for state dynamics between grid points. `EuropeanPricer` uses them automatically.
- **Euler / log-Euler / Milstein** live in the Rust crate only, not behind a Python engine switch.
- For production ATM Europeans, tune **`num_paths`** (and **variance reduction** elsewhere) before chasing ultra-fine **`num_steps`** under exact GBM.
- When you *do* use approximate schemes (other models), expect a **bias vs step size** trade-off: smaller **$ \Delta t $** costs more CPU for the same **paths**.

## Analyst lesson 2.6 — Why a symmetric payoff can make pairing worse


In [ ]:
import numpy as np
from finstack_quant.models.monte_carlo import simulate_gbm_paths

sample = simulate_gbm_paths(100, 0.05, 0, 0.2, 1, 1, 8192, seed=23)
terminal = np.asarray(sample.paths)[:, -1]
z = (np.log(terminal / 100) - (0.05 - 0.5 * 0.2**2)) / 0.2
# g(z)=z^2 is unchanged under z -> -z. Two reflected trajectories yield only one observation.
plain_payoffs = z**2
pair_means = (z[:4096]**2 + (-z[:4096])**2) / 2
assert np.array_equal(pair_means, z[:4096]**2)
plain_se = plain_payoffs.std(ddof=1) / np.sqrt(8192)
paired_se = pair_means.std(ddof=1) / np.sqrt(4096)
assert paired_se > plain_se
print(f"Same 8192 raw-path budget: plain SE={plain_se:.5f}, paired SE={paired_se:.5f}")
print(f"Observed ratio={paired_se/plain_se:.3f}; population prediction=sqrt(2)")
